# 06b-c supplement — confronto one-step information-matched

Correzione definitiva del confronto richiesto dal professore. Branch-ELM core e HayFlow voltage bridge ricevono **lo stesso identico tensore numerico** costruito da `S_t`, voltaggio e contesto assiale, ioni, morfologia e `U_realized`. Entrambi predicono la transizione autentica NEURON `V_(t+1)-V_t`. Non viene eseguito alcun rollout, `V_(t+1)` non entra negli input e il target non viene clippato.

In [ ]:
import importlib,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main');ROOT=Path('/kaggle/working');ELM_REPO=ROOT/'hayflow_workspace'/'elmneuron';ELM_REPO.parent.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO);REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();sys.path.insert(0,str(ELM_REPO));importlib.invalidate_caches();print({'revision':REVISION})

## 1. Input verificati

Servono il dataset targeted base, il top-up BAP v3, `hayflow_consolidated_autoregressive_go_no_go` (05t) e `hayflow_optimized_explicit_state_updater_canary` (06b). Non servono più H2, 05j-n o il fresh test 05j-o: appartenevano al confronto ritirato.

In [ ]:
from src.hayflow_model.rollout_aware_architecture_canary import discover_indexed_artifact_source
from src.hayflow_model.atomic_state_dynamics_playground import EXPECTED_05T_INDEX_SHA256
from src.hayflow_model import EXPECTED_06B_INDEX_SHA256
INPUT_ROOT=Path('/kaggle/input')
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_stamp';stamp=f'{source.stat().st_size}:{source.stat().st_mtime_ns}'
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
topup_override=os.environ.get('HAYFLOW_TOPUP_V3');topup_candidates=([Path(topup_override).expanduser()] if topup_override else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow06bc_matched_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifest_candidates)==1,manifest_candidates;COMPOSITE_MANIFEST=manifest_candidates[0]
base_override=os.environ.get('HAYFLOW_BASE_DATASET');base_candidates=([Path(base_override).expanduser()] if base_override else [])+[p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]+[p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None);assert BASE_SOURCE is not None,'Dataset base targeted v1.1 non trovato.'
source_05t_override=os.environ.get('HAYFLOW_05T_ARTIFACT');SOURCE_05T=discover_indexed_artifact_source(INPUT_ROOT,EXPECTED_05T_INDEX_SHA256,override=Path(source_05t_override) if source_05t_override else None);assert SOURCE_05T is not None,'Artefatto 05t esatto non trovato.'
source_06b_override=os.environ.get('HAYFLOW_06B_ARTIFACT');SOURCE_06B=discover_indexed_artifact_source(INPUT_ROOT,EXPECTED_06B_INDEX_SHA256,override=Path(source_06b_override) if source_06b_override else None);assert SOURCE_06B is not None,'Artefatto 06b optimized explicit state updater esatto non trovato.'
print({'base':str(BASE_SOURCE),'composite_manifest':str(COMPOSITE_MANIFEST),'05t':str(SOURCE_05T),'06b':str(SOURCE_06B)})

In [ ]:
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started={};hash_last={}
def hash_progress(name,done,total):
 now=time.monotonic();hash_started.setdefault(name,now);percent=int(100*done/total)
 if percent>=hash_last.get(name,-10)+10 or done==total:
  elapsed=now-hash_started[name];rate=done/max(elapsed,1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow ELM matched][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min',flush=True);hash_last[name]=percent
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=hash_progress);assert bundle.manifest['valid'] and bundle.transition_count==29880;print({'dataset_valid':True,'transitions':bundle.transition_count,'fingerprint':bundle.fingerprint})

## 2. Preflight del confronto

Il preflight deve confermare: stesso tensore per entrambi, target NEURON grezzo, nessun teacher endpoint negli input, nessun rollout, Branch-ELM da 8.002 parametri e voltage bridge da 8.985. Il sistema compatto completo conta anche il downstream STATE updater da 7.212 parametri, ma quest'ultimo non influenza la metrica del voltaggio.

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_model import CausalVoltageStateCouplingConfig,InformationMatchedTransitionConfig,InformationMatchedVoltageTransitionBenchmark
coupling_values=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_causal_voltage_state_coupling_forensic.yml').read_text())['causal_voltage_state_coupling_forensic'];coupling_config=CausalVoltageStateCouplingConfig.from_mapping(coupling_values)
comparison_values=yaml.safe_load((ELM_REPO/'configs/hayflow/branch_elm_information_matched_transition.yml').read_text())['information_matched_voltage_transition'];comparison_config=InformationMatchedTransitionConfig.from_mapping(comparison_values)
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_branch_elm_enriched_benchmark')
if OUTPUT_DIR.exists():
 assert not (OUTPUT_DIR/'final_report.json').exists(),f'Risultato completo gia presente: {OUTPUT_DIR}. Avvia una sessione nuova per non sovrascriverlo.'
 shutil.rmtree(OUTPUT_DIR);print({'stale_incomplete_output_removed':str(OUTPUT_DIR)})
session=InformationMatchedVoltageTransitionBenchmark(bundle,OUTPUT_DIR,coupling_config,comparison_config,SOURCE_05T,SOURCE_06B,code_revision=REVISION);contract=session.prepare_information_matched_benchmark()
display({'valid':contract['valid'],'comparison':contract['comparison_kind'],'same_numeric_input_tensor':contract['common_numeric_input_tensor'],'input_width':contract['common_input_width'],'input_fields':contract['common_input_fields'],'target':contract['target'],'target_clipping':contract['target_clipping'],'rollout':contract['autoregressive_rollout_performed'],'teacher_endpoint_input':contract['teacher_endpoint_used_as_input'],'models':contract['models'],'roles':contract['role_transition_counts']});assert contract['valid'] and contract['common_numeric_input_tensor'] and not contract['autoregressive_rollout_performed'] and not contract['teacher_endpoint_used_as_input']

In [ ]:
try:
 results=session.run_information_matched_benchmark();final_report=session.finalize_information_matched_benchmark(contract,results)
finally:
 session.close()
compact={name:{'global_rmse_mv':round(row['median_development_raw_endpoint_voltage_rmse_mv'],4),'soma_rmse_mv':round(row['median_development_raw_soma_endpoint_voltage_rmse_mv'],4)} for name,row in final_report['summary'].items()}
paired={seed:{key:round(value,4) for key,value in row.items()} for seed,row in final_report['paired_comparison'].items()}
display({'valid':final_report['valid'],'status':final_report['status'],'scope':final_report['comparison_scope'],'median_metrics':compact,'paired_hayflow_reduction_vs_elm':paired,'same_input':final_report['contract']['same_numeric_input_tensor'],'same_target':final_report['contract']['same_authentic_teacher_target'],'rollout':final_report['contract']['autoregressive_rollout_performed'],'sidecar_closed':final_report['professor_sidecar_closed_after_this_result'],'next_step':final_report['next_step']});assert final_report['valid'] and final_report['comparison_complete'] and final_report['scientific_voltage_ranking_authorized'] and final_report['professor_sidecar_closed_after_this_result']

## 3. Crea e scarica lo ZIP

Il downloader ricostruisce lo ZIP nel browser senza `FileLink` e senza stampare array o checkpoint.

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_branch_elm_enriched_benchmark','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,2),'download':'avviato dal browser'})